# 🤲 netctl_explore — the hands (M3.2)

Phase 2 proved *you* can drive the router; this notebook is the moment **code** can.
`GnmiProvisioner` replays M2.2's policer recipe over gNMI with one call — and, the deeper
point, it fills the exact hole `MockProvisioner` fills. Both satisfy the
`NetworkProvisioner` Protocol and pass ONE shared contract suite (rule 7): the controller
that gets wired in Phase 4 will not be able to tell which one it's holding. That
swappability is the whole architecture, demonstrated here with your own eyes.

**Prerequisites:** the lab is up (`containerlab deploy -t netlab/topology.clab.yml`).
**Companions:** [`docs/07-netlab.md`](../../docs/07-netlab.md) §6 (the recipe by hand) ·
[`docs/adr/006-lab-datapath-enforcement.md`](../../docs/adr/006-lab-datapath-enforcement.md)
(why a shim exists) · `netctl/tests/test_provisioner_contract.py` (these cells, as pinned
tests).

## 1. Two provisioners, one Protocol

The mock records; the real one configures routers. Nothing else may differ at the port.

In [1]:
from a2a_interfaces import NetworkProvisioner
from a2a_interfaces.fixtures import CAPACITY_50_MBPS, QOS_CLASS, RESOLVED_PATH

from netctl.mock import MockProvisioner
from netctl.connect import GnmiTarget
from netctl.provisioner import GnmiProvisioner
from netctl.testing import lab_ipv4

mock = MockProvisioner()
ip = lab_ipv4()
assert ip, "the lab isn't running — containerlab deploy -t netlab/topology.clab.yml"
real = GnmiProvisioner({"srl1": GnmiTarget(host=ip, tls_name="srl1")})

for name, prov in [("mock", mock), ("gnmi", real)]:
    print(f"{name:5} satisfies NetworkProvisioner: {isinstance(prov, NetworkProvisioner)}   health: {prov.health()}")


mock  satisfies NetworkProvisioner: True   health: True
gnmi  satisfies NetworkProvisioner: True   health: True


## 2. The canonical call — Ada's 50 Mbps, as one function

`RESOLVED_PATH` is what the controller will hand over after resolving ticket #7's
`resourceId` (ADR-005: netctl never sees resource ids, only concrete device+interfaces).
Watch the same call hit both provisioners:

In [2]:
SESSION = "nb-session"

for name, prov in [("mock", mock), ("gnmi", real)]:
    result = prov.apply_bandwidth(SESSION, RESOLVED_PATH, CAPACITY_50_MBPS, QOS_CLASS)
    print(f"{name:5} → ok={result.ok}  {result.detail}")

print("\nwhat the mock remembers:", mock.applied[SESSION])


mock  → ok=True  
gnmi  → ok=True  policer a2a-nb-session @ srl1/ethernet-1/1.0

what the mock remembers: {'path': ResolvedPath(device='srl1', ingress_if='ethernet-1/1', egress_if='ethernet-1/2'), 'capacity_bps': 50000000, 'qos_class': 1}


## 3. What actually landed on the router

Read the committed config back over gNMI — the policer template named after the session
(`a2a-nb-session`, the naming that makes stateless teardown possible) and its attachment.
Note the `srl_nokia-acl-policers:` module prefix on the attachment: RFC 7951 namespaces
keys that come from YANG *augments*, one of this slice's gotchas.

In [3]:
import json

client = real._client("srl1")
raw = client.get(path=["/qos"], encoding="json_ietf", datatype="config")
print(json.dumps(raw["notification"][0]["update"][0]["val"], indent=1)[:900])


{
 "interfaces": {
  "interface": [
   {
    "interface-id": "ethernet-1/1.0",
    "interface-ref": {
     "interface": "ethernet-1/1",
     "subinterface": 0
    },
    "input": {
     "srl_nokia-acl-policers:policer-templates": {
      "policer-template": "a2a-nb-session"
     }
    }
   }
  ]
 },
 "srl_nokia-acl-policers:policer-templates": {
  "policer-template": [
   {
    "name": "a2a-ent7-a0",
    "policer": [
     {
      "sequence-id": 1,
      "peak-rate-kbps": 50000,
      "committed-rate-kbps": 50000,
      "maximum-burst-size": 125000,
      "committed-burst-size": 125000,
      "violate-action": {
       "drop": [
        null
       ]
      }
     }
    ]
   },
   {
    "name": "a2a-nb-session",
    "policer": [
     {
      "sequence-id": 1,
      "peak-rate-kbps": 50000,
      "committed-rate-kbps": 50000,
      "maximum-burst-size": 125000,
      "committed-burst-size":


## 4. Physics check: the plateau, from one call

The config is committed, but this lab's datapath needs its missing ASIC
(ADR-006): one shim tick materializes the committed policer as `tc tbf` inside srl1's
netns. Then offer 100 Mbit/s of UDP and watch what survives:

In [4]:
import subprocess
from pathlib import Path

SHIM = Path.cwd().parents[1] / "netlab" / "mirror-policer-to-tc.sh"

def shim_tick():
    print(subprocess.run([str(SHIM)], capture_output=True, text=True).stdout.strip())

def offered_100M_received_mbps():
    subprocess.run(["docker","exec","-d","clab-a2a-hostB","iperf3","-s","-p","5220","-1"], check=False)
    out = subprocess.run(
        ["docker","exec","clab-a2a-hostA","iperf3","-c","10.10.2.10","-p","5220","-t","5","-u","-b","100M","--json"],
        capture_output=True, text=True, check=True).stdout
    s = json.loads(out)["end"]["sum"]
    return s["bits_per_second"] * (1 - s["lost_percent"]/100) / 1e6

shim_tick()
shaped = offered_100M_received_mbps()
print(f"offered 100 Mbit/s → received {shaped:.1f} Mbit/s   ← Ada's plateau")
assert 40 < shaped < 55


shim: tbf 50000kbit on clab-a2a-srl1/e1-2 (mirroring the committed policer)


offered 100 Mbit/s → received 49.0 Mbit/s   ← Ada's plateau


## 5. Teardown — twice, because rule 8 says so

Teardown is *stateless*: the provisioner FINDS its work on the router by the session
naming, so a second call (or a call from a freshly restarted process) is the same
success. Then the physics returns to the ceiling:

In [5]:
print("first :", real.teardown(SESSION))
print("second:", real.teardown(SESSION), "  ← idempotent, not an error")

shim_tick()
unshaped = offered_100M_received_mbps()
print(f"offered 100 Mbit/s → received {unshaped:.1f} Mbit/s   ← ceiling is back")
assert unshaped > 85

mock.teardown(SESSION)
print("mock after teardown:", mock.applied, mock.torn_down)
real.close()


first : ok=True detail='removed: srl1'
second: ok=True detail='nothing to remove'   ← idempotent, not an error


shim: no policer attached — shaper removed


offered 100 Mbit/s → received 100.0 Mbit/s   ← ceiling is back
mock after teardown: {} ['nb-session']


## 6. The second product: telemetry — the RIGHT to configure the device (M3.3, ADR-007)

The telemetry ticket doesn't ship you data — it's **the right to push a telemetry config
to the router**. So `apply_telemetry` is *symmetric* with `apply_bandwidth`: it writes a
real gNMI dial-out destination (SR Linux `grpc-tunnel destination`) pointing at your
collector, and — exactly like the policer — you read it straight back off the device, and
teardown removes it. The token **is** the access. (This replaced an earlier provider-side
forwarder; see ADR-007's "Revision".)

In [6]:
from a2a_interfaces.models import ResolvedNode

real2 = GnmiProvisioner({"srl1": GnmiTarget(host=ip, tls_name="srl1")})

# the ticket authorizes ONE config write: a telemetry export destination → your collector
result = real2.apply_telemetry(
    "nb-telemetry", ResolvedNode(device="srl1"),
    ["/interface[name=ethernet-1/1]/statistics"],
    "10.0.0.50:57400", sample_interval_s=10,
)
print(result.detail)

# read it back off the router — proof the ticket's right was exercised ON the device
print("\non srl1 now:", real2.telemetry_config("srl1"))

# ...and teardown removes it, statelessly (rule 8), just like the policer
print("\nteardown:", real2.teardown("nb-telemetry").detail)
print("on srl1 after:", real2.telemetry_config("srl1"), "  ← removed")
real2.close()

telemetry export a2a-nb-telemetry @ srl1 → 10.0.0.50:57400

on srl1 now: [{'name': 'a2a-nb-telemetry', 'address': '10.0.0.50', 'port': 57400, 'network-instance': 'mgmt'}]

teardown: removed: srl1
on srl1 after: []   ← removed


## What you learned (and where it goes)

| You did | The concept | Next |
|---|---|---|
| one call on two provisioners | rule 7: mock parity at the port | the controller can't tell (M4.5) |
| read `a2a-nb-session` off the router | sessions leave names, not process state | restart-safe teardown |
| module-prefixed JSON keys | RFC 7951 + YANG augments | `_denamespace` in provisioner.py |
| shim tick → plateau → teardown → ceiling | ADR-006's missing ASIC | the lab fixture loops it (M3.4) |
| teardown twice, both ok | rule 8, idempotency | revocation can't double-fault (M4.5) |

## Experiments to try

- Apply with `capacity_bps=20_000_000`, shim, re-measure — the plateau follows the ticket.
- Apply two sessions on the same subinterface — what happens, and is that the right
  answer for the E_CONFLICT check the controller runs (M4.1)?
- Kill the provisioner object after `apply`, build a fresh one, `teardown` — it still
  cleans up. Where did the state live the whole time?
- Read `real._session_config_on(real._client("srl1"), "a2a-ghost")` — the empty list is
  why unknown-session teardown is a success, not an error.